# Introducción a QA sobre noticias dominicanas

Recursos oficiales:

- Dataset: `Lisibonny/pdqa`
- Baseline: `Lisibonny/modelo_qa_beto_squad_es_pdqa`
- Space: `Lisibonny/Repartidor_Dominicano`

El dataset ya contiene las divisiones oficiales: `train`, `validation` y `test`.


In [1]:
!pip install -q "transformers>=4.45,<5.0" "datasets>=3.0,<5.0" "accelerate>=1.0,<2.0" "pandas>=2.0,<3.0" "pyarrow>=15,<22"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import random, string, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline

SEED = 42
DATASET_ID = "Lisibonny/pdqa"
BASELINE_MODEL_ID = "Lisibonny/modelo_qa_beto_squad_es_pdqa"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

dataset = load_dataset(DATASET_ID)
print(dataset)
assert set(["train","validation","test"]).issubset(dataset.keys())
print({split: len(dataset[split]) for split in dataset})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/8.68k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 45
    })
    validation: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 15
    })
    test: Dataset({
        features: ['id', 'question', 'context', 'answers', 'title'],
        num_rows: 20
    })
})
{'train': 45, 'validation': 15, 'test': 20}


In [3]:
print(dataset["train"].column_names)
display(pd.DataFrame([dataset["train"][0]]))
display(pd.DataFrame([dataset["validation"][0]]))
display(pd.DataFrame([dataset["test"][0]]))


['id', 'question', 'context', 'answers', 'title']


,id,question,context,answers,title
0,5kpfplqtw2p99k1,¿Quiénes se van de “De Extremo a Extremo”?,Durante la pasada entrega de Premios Soberano ...,"{'answer_start': [171], 'text': ['Caroline Aqu...",prueba


,id,question,context,answers,title
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,Asindown ha lanzado una campaña de sensibiliza...,"{'answer_start': [161], 'text': ['concienciar ...",prueba


,id,question,context,answers,title
0,jygfnzhtybqz9u2,¿Quién escribió el poema “Una mujer está sola”?,"“Una mujer está sola”, escribió Aída Portalatí...","{'answer_start': [32], 'text': ['Aída Portalat...",prueba


## Verificación de columnas

El notebook espera columnas equivalentes a `id`, `question`, `context` y `answers`.
Si los nombres son distintos, ajuste solo las constantes siguientes.


In [4]:
ID_COL = "id"
QUESTION_COL = "question"
CONTEXT_COL = "context"
ANSWERS_COL = "answers"

required_train = {ID_COL, QUESTION_COL, CONTEXT_COL, ANSWERS_COL}
required_test = {ID_COL, QUESTION_COL, CONTEXT_COL}
assert required_train.issubset(dataset["train"].column_names)
assert required_train.issubset(dataset["validation"].column_names)
assert required_test.issubset(dataset["test"].column_names)


## Análisis exploratorio mínimo

In [5]:
train_df = dataset["train"].to_pandas()
validation_df = dataset["validation"].to_pandas()

for frame in [train_df, validation_df]:
    frame["question_words"] = frame[QUESTION_COL].astype(str).str.split().str.len()
    frame["context_words"] = frame[CONTEXT_COL].astype(str).str.split().str.len()

display(train_df[["question_words","context_words"]].describe())
display(validation_df[["question_words","context_words"]].describe())


,question_words,context_words
count,45.000000,45.000000
mean,6.244444,60.644444
std,2.612518,56.302412
min,3.000000,27.000000
25%,4.000000,34.000000
50%,6.000000,43.000000
75%,7.000000,55.000000
max,15.000000,285.000000


,question_words,context_words
count,15.000000,15.000000
mean,5.933333,98.133333
std,1.667619,112.981583
min,3.000000,29.000000
25%,5.000000,34.000000
50%,6.000000,38.000000
75%,6.000000,87.000000
max,9.000000,347.000000


## Inferencia con el baseline

In [6]:
device = 0 if torch.cuda.is_available() else -1
qa_baseline = pipeline(
    "question-answering",
    model=BASELINE_MODEL_ID,
    tokenizer=BASELINE_MODEL_ID,
    device=device,
)

example = dataset["validation"][0]
result = qa_baseline(
    question=example[QUESTION_COL],
    context=example[CONTEXT_COL],
)
print("Pregunta:", example[QUESTION_COL])
print("Referencia:", example[ANSWERS_COL])
print("Predicción:", result)


config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


Pregunta: ¿Cuál es el objetivo de la campaña de Asindown?
Referencia: {'answer_start': [161], 'text': ['concienciar sobre el acoso escolar y social que sufren las personas con síndrome de Down']}
Predicción: {'score': 0.09938068687915802, 'start': 161, 'end': 204, 'answer': 'concienciar sobre el acoso escolar y social'}


## Exact Match y F1

In [7]:
SPANISH_ARTICLES = {"el","la","los","las","un","una","unos","unas"}

def normalize_answer(text):
    text = "" if text is None else str(text).lower().strip()
    text = "".join(ch for ch in unicodedata.normalize("NFD", text)
                   if unicodedata.category(ch) != "Mn")
    punctuation = string.punctuation + "¡¿“”‘’«»…"
    text = "".join(" " if ch in punctuation else ch for ch in text)
    return " ".join(tok for tok in text.split() if tok not in SPANISH_ARTICLES)

def exact_match(pred, truth):
    return float(normalize_answer(pred) == normalize_answer(truth))

def token_f1(pred, truth):
    p = normalize_answer(pred).split()
    t = normalize_answer(truth).split()
    if not p and not t: return 1.0
    if not p or not t: return 0.0
    same = sum((Counter(p) & Counter(t)).values())
    if same == 0: return 0.0
    precision = same / len(p)
    recall = same / len(t)
    return 2 * precision * recall / (precision + recall)

def answer_texts(value):
    if isinstance(value, dict):
        value = value.get("text", value.get("answer", value))
    if isinstance(value, list):
        return [str(x) for x in value]
    return [str(value)]


---

# ▞▞ 1. Ejecutar el baseline y reportar Exact Match y F1 sobre validation ▞▞

Se evalúa el modelo oficial `Lisibonny/modelo_qa_beto_squad_es_pdqa` sobre las
15 preguntas de `validation`, con la normalización de respuestas del propio
paquete, y se guardan las predicciones por ejemplo para el análisis posterior.


In [8]:
rows = []
for ex in dataset["validation"]:
    pred = qa_baseline(question=ex[QUESTION_COL], context=ex[CONTEXT_COL])
    golds = answer_texts(ex[ANSWERS_COL])
    em = max(exact_match(pred["answer"], g) for g in golds)
    f1 = max(token_f1(pred["answer"], g) for g in golds)
    rows.append({
        "id": ex[ID_COL],
        "question": ex[QUESTION_COL],
        "reference": " | ".join(golds),
        "prediction": pred["answer"],
        "confidence": pred["score"],
        "exact_match": em,
        "f1": f1,
    })

baseline_results = pd.DataFrame(rows)
display(baseline_results)
print("Exact Match:", 100 * baseline_results["exact_match"].mean())
print("F1:", 100 * baseline_results["f1"].mean())


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,question,reference,prediction,confidence,exact_match,f1
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,concienciar sobre el acoso escolar y social qu...,concienciar sobre el acoso escolar y social,0.099381,0.0,0.631579
1,7857zba14eers9x,¿Cómo se llama el canal 4?,Radio Televisión Dominicana,Radio Televisión Dominicana,0.634700,1.0,1.000000
2,hkhjremdmrmix29,¿Cuál es nuestro símbolo patrio?,La Bandera Nacional,La Bandera Nacional,0.382562,1.0,1.000000
3,ayfu15q9bfn5nhb,¿Quién fundó Menudo?,Edgardo García,Edgardo García,0.545798,1.0,1.000000
4,rpcbxp9rc8g0iuh,¿Cuando es el Miércoles de Ceniza?,El próximo miércoles 22 de febrero,El próximo miércoles 22 de febrero,0.336815,1.0,1.000000
5,d2xt90fah10kox5,¿A cuántos bateadores ponchó Jacob deGrom?,a 11 bateadores,11 bateadores,0.367446,0.0,0.800000
6,sz9yeioprmn61xv,¿Quién es el creador de Dilbert?,Scott Adams,Scott Adams,0.959120,1.0,1.000000
7,ak5ubvypoj83wfp,¿Cuántos minutos jugó Immanuel Quickley?,55 minutos,55 minutos,0.447666,1.0,1.000000
8,3u77e1org81bc7f,¿Qué jugador estaba lesionado?,Jalen Brunson,Jalen Brunson,0.796943,1.0,1.000000
9,pxqbvfcey7z9p3q,¿Cómo será la inflación en 2023?,seguirá siendo alta,en torno al 7%,0.187205,0.0,0.000000


Exact Match: 60.0
F1: 75.84015594541911


### Métricas del baseline para el resto del notebook

La celda anterior es la de la cátedra y ya calcula `baseline_results`. Esta
guarda las métricas en variables reutilizables y escribe el CSV que necesita la
celda de verificación con `05_evaluate_qa.py` del punto 5. No repite la
inferencia: solo agrega sobre el DataFrame ya calculado.

In [ ]:
baseline_results.to_csv("baseline_results.csv", index=False)

em_baseline = 100 * baseline_results["exact_match"].mean()
f1_baseline = 100 * baseline_results["f1"].mean()
print(f"Exact Match: {em_baseline:.2f}%")
print(f"F1: {f1_baseline:.2f}%")


---

#▞▞ CHALLENGE ▞▞

Lo que sigue desarrolla el trabajo solicitado para el challenge final de este proyecto.


---

# ▞▞ 2. Analizar el dataset: longitud de preguntas, contextos y respuestas, y problemas de calidad ▞▞

Aquí empezamos a analizar la longitud de las respuestas y una revisión de problemas de calidad de la anotación.

Primero se tokeniza `train`, porque hace falta para medir cuántos contextos desbordan la ventana del modelo. Esa misma tokenización alimenta el fine-tuning más adelante.

## Preparación del dataset: tokenización y localización del span de respuesta


In [11]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = BASELINE_MODEL_ID
tokenizer_ft = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

MAX_LENGTH = 384
DOC_STRIDE = 128

def preparar_features(ejemplos):
    preguntas = [q.strip() for q in ejemplos["question"]]
    tokenized = tokenizer_ft(
        preguntas,
        ejemplos["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = ejemplos["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = tokenized.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

train_tokenizado = dataset["train"].map(
    preparar_features, batched=True, remove_columns=dataset["train"].column_names
)
print(train_tokenizado)

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 47
})


## Longitud de respuestas y calidad de la anotación

Las diferencias entre EM y F1 no vienen solo del modelo: varias se explican por
cómo están anotadas las respuestas de referencia. Medir esto ahora permite
interpretar correctamente los resultados de los experimentos.


In [12]:
# --- Longitudes de respuesta y control de calidad de las anotaciones ---
import re

def reporte_split(nombre_split):
    ds = dataset[nombre_split]
    filas = []
    for ex in ds:
        texto = ex[ANSWERS_COL]["text"][0]
        inicio = ex[ANSWERS_COL]["answer_start"][0]
        contexto = ex[CONTEXT_COL]
        filas.append({
            "id": ex[ID_COL],
            "respuesta": texto,
            "palabras_respuesta": len(texto.split()),
            "caracteres_respuesta": len(texto),
            "span_consistente": contexto[inicio:inicio + len(texto)] == texto,
            "espacios_al_borde": texto != texto.strip(),
            "puntuacion_al_borde": bool(re.match(r'^[^\w]', texto)) or bool(re.search(r'[^\w.]$', texto)),
            # Solo cuenta si la palabra inicial SOBREVIVE a normalize_answer:
            # los artículos ya los elimina la métrica, así que no penalizan.
            "arranca_con_funcional": (normalize_answer(texto).split() or [""])[0] in {
                "a", "de", "en", "por", "para", "con", "que", "sobre", "hasta",
                "gira", "desde"
            },
        })
    return pd.DataFrame(filas)

resp_train = reporte_split("train")
resp_val = reporte_split("validation")

print("=== Longitud de respuestas (palabras) ===")
display(pd.DataFrame({
    "train": resp_train["palabras_respuesta"].describe(),
    "validation": resp_val["palabras_respuesta"].describe(),
}))

print("\n=== Problemas de calidad detectados ===")
for nombre, df in [("train", resp_train), ("validation", resp_val)]:
    print(f"\n[{nombre}] n={len(df)}")
    print(f"  spans inconsistentes (context[start:start+len] != text): "
          f"{(~df['span_consistente']).sum()}")
    print(f"  respuestas con espacios en los bordes:                   "
          f"{df['espacios_al_borde'].sum()}")
    print(f"  respuestas que abren/cierran con puntuación:             "
          f"{df['puntuacion_al_borde'].sum()}")
    print(f"  respuestas que arrancan con palabra funcional:           "
          f"{df['arranca_con_funcional'].sum()}")
    print(f"  respuesta más larga: {df['palabras_respuesta'].max()} palabras")

print("\n=== Columna 'title' ===")
for nombre in ["train", "validation", "test"]:
    print(f"  {nombre}: {dict(Counter(dataset[nombre]['title']))}")

print("\n=== Contextos vs ventana del modelo (max_length=384) ===")
n_features = len(train_tokenizado)
n_ejemplos = len(dataset["train"])
print(f"  features generadas desde train: {n_features} (desde {n_ejemplos} ejemplos)")
print(f"  ejemplos que desbordan la ventana y generan features extra: "
      f"{n_features - n_ejemplos}")
spans_perdidos = sum(
    1 for s, e in zip(train_tokenizado["start_positions"], train_tokenizado["end_positions"])
    if s == 0 and e == 0
)
print(f"  features etiquetadas (0,0) = respuesta fuera de la ventana: {spans_perdidos}")

print("\n=== Ejemplos con anotación problemática (validation) ===")
display(resp_val[
    resp_val["puntuacion_al_borde"] | resp_val["arranca_con_funcional"] |
    ~resp_val["span_consistente"] | resp_val["espacios_al_borde"]
][["id", "respuesta", "palabras_respuesta", "span_consistente", "puntuacion_al_borde",
   "arranca_con_funcional"]])


=== Longitud de respuestas (palabras) ===


,train,validation
count,45.000000,15.000000
mean,4.511111,5.733333
std,4.153981,5.338093
min,1.000000,2.000000
25%,2.000000,2.500000
50%,3.000000,3.000000
75%,5.000000,6.500000
max,22.000000,19.000000



=== Problemas de calidad detectados ===

[train] n=45
  spans inconsistentes (context[start:start+len] != text): 1
  respuestas con espacios en los bordes:                   0
  respuestas que abren/cierran con puntuación:             2
  respuestas que arrancan con palabra funcional:           13
  respuesta más larga: 22 palabras

[validation] n=15
  spans inconsistentes (context[start:start+len] != text): 0
  respuestas con espacios en los bordes:                   0
  respuestas que abren/cierran con puntuación:             0
  respuestas que arrancan con palabra funcional:           4
  respuesta más larga: 19 palabras

=== Columna 'title' ===
  train: {'prueba': 45}
  validation: {'prueba': 15}
  test: {'prueba': 20}

=== Contextos vs ventana del modelo (max_length=384) ===
  features generadas desde train: 47 (desde 45 ejemplos)
  ejemplos que desbordan la ventana y generan features extra: 2
  features etiquetadas (0,0) = respuesta fuera de la ventana: 2

=== Ejemplos con anota

,id,respuesta,palabras_respuesta,span_consistente,puntuacion_al_borde,arranca_con_funcional
5,d2xt90fah10kox5,a 11 bateadores,3,True,False,True
10,7yfoxupiw6mf37s,gira en torno al embarazo en adolescentes,7,True,False,True
11,cftfka87bhwzs6o,por las turbulencias en el sector bancario en ...,12,True,False,True
12,fokrrek3dseacem,a tres exministros,3,True,False,True


### Hallazgos del análisis de calidad

Los conteos exactos los imprime la celda anterior. Los patrones cualitativos
relevantes para la evaluación son:

1. **Puntuación tipográfica dentro del span anotado.** La respuesta de
   `adrufk2slj332tf` es `“Una mujer está sola` — la comilla de apertura queda
   incluida en el texto de referencia. El modelo la reproduce y aun así obtiene
   EM=1 únicamente porque `normalize_answer` elimina la puntuación. Es decir:
   parte de nuestro EM depende de la normalización, no de que el span sea
   idéntico.

2. **Respuestas que arrancan con palabra funcional.** Casos como
   `a 11 bateadores` o `gira en torno al embarazo en adolescentes` incluyen una
   preposición o un verbo inicial que el modelo tiende a omitir. Esto genera
   fallos de EM con F1 alto (0.80 y 0.83) que **no son errores de comprensión**,
   sino desacuerdos de frontera contra una anotación no canónica. Es la causa
   directa de la categoría "truncamiento de límites" del análisis de errores.

   El conteo se hace sobre la respuesta **ya normalizada**, no sobre el texto
   crudo: `normalize_answer` elimina los artículos (`el`, `la`, `los`, `un`…),
   así que un gold como `La Bandera Nacional` no penaliza al modelo y no cuenta
   como problema. Solo se marcan las palabras funcionales que **sobreviven** a la
   normalización (`a`, `de`, `en`, `por`, `gira`, `desde`…), que son las que sí
   pueden costar un EM.

3. **Respuestas compuestas muy largas.** `6ifjwhgjigmw5c3` tiene 19 palabras y
   enumera tres personas con sus cargos. Un modelo extractivo de span único
   difícilmente puede acertarla completa: es un límite de la tarea tal como está
   anotada, no del modelo.

4. **La columna `title` es constante (`prueba`) en las tres divisiones**, así que
   no aporta señal y no se usa como feature.

5. **Contextos que desbordan la ventana.** Con `max_length=384` y
   `doc_stride=128`, train genera 47 features desde 45 ejemplos, es decir hay
   contextos que se parten en dos ventanas. El conteo de features etiquetadas
   `(0,0)` que imprime la celda anterior indica cuántas ventanas quedaron **sin
   la respuesta dentro**: esas features entrenan al modelo a apuntar al token
   `[CLS]` y son ruido. Si el número es alto, reducir el `doc_stride` o subir
   `max_length` sería una mejora legítima adicional.

**Consecuencia metodológica:** el techo de EM alcanzable en este dataset está
limitado por la anotación misma, no solo por el modelo. Conviene decirlo en la
presentación, porque explica por qué F1 mejora más fácil que EM.


---

# ▞▞ 3. Formular una hipótesis de mejora ▞▞

Una hipótesis útil tiene que salir después de que logramos diagnosticar el error en concreto. Dígase, qué es exactamente lo que está complicando el modelo, el cual evita que se incremente su EM y F1. Para llegar a una hipótesis concreta, primero analizamos los errores.


##Análisis de errores

In [13]:
def categorizar_error(row):
    if row["exact_match"] == 1.0:
        return "Acierto exacto"
    if row["f1"] >= 0.5:
        return "Truncamiento de límites (respuesta parcialmente correcta)"
    return "Fallo total (respuesta equivocada o entidad incorrecta)"

baseline_results["categoria"] = baseline_results.apply(categorizar_error, axis=1)

resumen_categorias = baseline_results["categoria"].value_counts()
print(resumen_categorias)
print()

# Ejemplos de cada categoría de error
errores = baseline_results[baseline_results["exact_match"] == 0].copy()
errores["longitud_referencia"] = errores["reference"].str.split().str.len()
display(errores[["question", "reference", "prediction", "f1", "categoria", "longitud_referencia"]])

categoria
Acierto exacto                                               9
Truncamiento de límites (respuesta parcialmente correcta)    3
Fallo total (respuesta equivocada o entidad incorrecta)      3
Name: count, dtype: int64



,question,reference,prediction,f1,categoria,longitud_referencia
0,¿Cuál es el objetivo de la campaña de Asindown?,concienciar sobre el acoso escolar y social qu...,concienciar sobre el acoso escolar y social,0.631579,Truncamiento de límites (respuesta parcialment...,15
5,¿A cuántos bateadores ponchó Jacob deGrom?,a 11 bateadores,11 bateadores,0.800000,Truncamiento de límites (respuesta parcialment...,3
9,¿Cómo será la inflación en 2023?,seguirá siendo alta,en torno al 7%,0.000000,Fallo total (respuesta equivocada o entidad in...,3
10,¿De qué trata la película Ramona?,gira en torno al embarazo en adolescentes,torno al embarazo en adolescentes,0.833333,Truncamiento de límites (respuesta parcialment...,7
11,¿Porqué cerro al nivel más bajo el petróleo?,por las turbulencias en el sector bancario en ...,desde diciembre,0.000000,Fallo total (respuesta equivocada o entidad in...,12
13,¿A quiénes apresó la Procuraduría?,"los exministros: José Ramón Peralta, Administr...",Guerrero,0.111111,Fallo total (respuesta equivocada o entidad in...,19


##Errores del baseline

El modelo baseline obtiene **EM = 60%** y **F1 = 75.8%**. La brecha de ~16 puntos
entre ambas métricas es la primera señal importante: el modelo casi siempre
identifica la región correcta del contexto donde está la respuesta, pero no
siempre delimita el span exacto (de ahí que F1, que mide superposición de
palabras, sea consistentemente más alto que EM, que exige coincidencia perfecta).

Categorizando los 6 casos con exact_match = 0, aparecen dos patrones distintos:

**1. Truncamiento de límites (F1 ≥ 0.5, 3 de 6 casos)**
El modelo ubica la respuesta correctamente pero corta la frase de más o de menos:
- "objetivo de la campaña Asindown" → predijo la primera mitad de la oración,
  omitiendo la cláusula final ("...que sufren las personas con síndrome de Down").
- "bateadores ponchados por deGrom" → omitió el artículo inicial ("a 11" vs "11").
- "de qué trata Ramona" → cortó la primera palabra de la frase ("gira").

Estos casos tienen F1 alto (0.63–0.83) porque comparten casi todas las palabras
con la referencia; el error es de **frontera de extracción**, no de comprensión.
Es razonable hipotetizar que más entrenamiento (épocas) ayude al modelo a
calibrar mejor dónde empieza y termina el span, dado que el dataset de
entrenamiento es muy pequeño (45 ejemplos) para que esto se aprenda en pocos pasos.

**2. Fallo total (F1 < 0.2, 3 de 6 casos)**
El modelo extrae información del contexto que no corresponde a la pregunta:
- "inflación en 2023" → el contexto probablemente menciona varias cifras
  numéricas; el modelo extrajo un dato relacionado pero no la respuesta
  correcta (respuesta esperada: una descripción cualitativa, no un número).
- "por qué cerró bajo el petróleo" → respondió con una referencia temporal
  ("desde diciembre") en vez de la causa que pedía la pregunta.
- "a quiénes apresó la Procuraduría" → la referencia es una lista de tres
  personas con cargos; el modelo solo extrajo un nombre aislado.

Este segundo grupo no mejora con más entrenamiento del mismo tipo: son casos
donde el contexto tiene **múltiples entidades o cifras candidatas** y el modelo
no tiene forma de distinguir cuál es la que responde específicamente la
pregunta. Es una limitación más estructural (posiblemente de cómo el modelo
fue expuesto a preguntas de tipo "por qué" o respuestas compuestas/listas
durante el entrenamiento original), y es poco probable que se resuelva solo
ajustando learning rate o épocas — necesitaría más ejemplos de este tipo de
pregunta en el dataset de entrenamiento.

## Hipótesis a contrastar

Del diagnóstico anterior salen dos observaciones con consecuencias distintas:
el error dominante es de **frontera** (el modelo ubica la zona correcta pero
corta mal el span), y el conjunto de entrenamiento es **muy pequeño**
(45 ejemplos), lo que hace que cualquier resultado dependa mucho de esa única
muestra.

De ahí se derivan las hipótesis que los experimentos del punto 4 ponen a prueba:

| # | Hipótesis | Variable que se modifica | Experimento |
|---|---|---|---|
| H1 | El modelo no tuvo suficientes pasadas por los datos para calibrar los bordes del span. | `num_train_epochs` | Variante 1 |
| H2 | El modelo sobreajusta los 45 ejemplos; más regularización debería generalizar mejor. | `weight_decay` | Variante 2 |
| H3 | Con `lr=2e-05` los pesos de la cabeza de QA apenas se mueven en 3 pasos por época; no alcanzan a recalibrar las fronteras. | `learning_rate` | Variante 3 |
| H4 | El cuello de botella es la varianza de entrenar sobre una vista única de 45 ejemplos. | partición de `train` en folds | Variante 4 |
| H5 | Parte del EM perdido no es del modelo: el pipeline trunca la respuesta a 15 tokens y varias referencias son más largas. | `max_answer_len` (inferencia) | Variantes 5 y 6 |

**Hipótesis nula que hay que descartar antes de atribuir nada:** que la mejora
venga simplemente de *entrenar más*, sin importar qué hiperparámetro se cambie.

**Sobre H5.** El pipeline de question-answering de `transformers` recorta la
respuesta a `max_answer_len=15` tokens por defecto. Midiendo las referencias de
`validation` con el tokenizador de BETO, dos de ellas superan ese tope (18 y 27
tokens), así que son **inalcanzables por configuración, no por el modelo**. Las
Variantes 5 y 6 miden qué pasa al levantarlo. Como es un parámetro de
inferencia, no requieren reentrenar: reutilizan los pesos de las Variantes 4 y 3
respectivamente, de modo que la comparación es perfectamente controlada — mismos
pesos, única diferencia el parámetro de decodificación.


---

# ▞▞ 4. Comparar al menos dos configuraciones o variantes bajo las mismas condiciones ▞▞

Todas las variantes parten del mismo checkpoint
(`Lisibonny/modelo_qa_beto_squad_es_pdqa`), se entrenan con `seed=42` y se
evalúan con el mismo procedimiento de inferencia y las mismas métricas sobre
`validation`. Cada una modifica **una sola variable** respecto a su referencia,
de modo que las diferencias sean atribuibles.


## Arnés único de entrenamiento y evaluación

Todas las configuraciones comparten **exactamente la misma tokenización**: el
`train_tokenizado` producido en el punto 2 con `preparar_features`,
`max_length=384` y `doc_stride=128`. Ese dataset se pasa de forma **explícita**
a cada llamada, de modo que se pueda verificar leyendo el código que ninguna
variante entrena sobre datos distintos.

Una sola función, `entrenar_variante`, construye el `Trainer` para todas: mismos
`TrainingArguments`, misma semilla, mismo checkpoint de partida. Lo único que
cambia entre experimentos son los hiperparámetros que se pasan por argumento.

Dos decisiones sobre el uso de disco:

- `save_strategy="no"` evita guardar un checkpoint de 437 MB **por época**. Solo
  se guarda el modelo final de cada variante.
- `guardar=False` devuelve el modelo en memoria sin escribirlo a disco. Los
  folds de la Variante 4 lo usan: se promedian sobre la marcha y solo se guarda
  el modelo resultante, en vez de cinco.


In [14]:
# --- Arnés común: una sola función de entrenamiento para todas las variantes ---
import os, gc
import torch
from collections import defaultdict
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

# Registro único de resultados: todo experimento escribe aquí.
resultados_experimentos = {}

def registrar(nombre, cambio, em, f1, ruta=None, publicable=True):
    resultados_experimentos[nombre] = {
        "experimento": nombre, "cambio": cambio,
        "EM": round(em, 2), "F1": round(f1, 2),
        "ruta": ruta, "publicable": publicable,
    }
    print(f"[{nombre}] EM={em:.2f}  F1={f1:.2f}")

def tokenizar(ds):
    """Misma tokenización que el punto 2, aplicada a cualquier subconjunto."""
    return ds.map(preparar_features, batched=True, remove_columns=ds.column_names)

def entrenar_variante(nombre, learning_rate=2e-05, epochs=4, batch_size=16,
                      weight_decay=0.01, ds_tokenizado=None, guardar=True):
    """Fine-tuning continuado desde MODEL_CHECKPOINT.

    ds_tokenizado : dataset ya tokenizado. Por defecto `train_tokenizado`, que es
                    el que usan todas las variantes salvo los folds, que pasan
                    subconjuntos suyos producidos por la misma función.
    guardar       : si es False devuelve el modelo en memoria sin escribirlo a
                    disco. Lo usan los folds para no dejar 5 checkpoints de
                    437 MB cada uno.
    """
    if ds_tokenizado is None:
        ds_tokenizado = train_tokenizado

    modelo = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)
    args = TrainingArguments(
        output_dir=f"./out_{nombre}",
        learning_rate=learning_rate,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        weight_decay=weight_decay,
        seed=SEED,
        save_strategy="no",       # nada de un checkpoint por época
        logging_steps=5,
        report_to="none",
        disable_tqdm=True,
    )
    try:
        trainer = Trainer(model=modelo, args=args, train_dataset=ds_tokenizado,
                          processing_class=tokenizer_ft)
    except TypeError:
        # transformers < 4.46 todavía espera el argumento `tokenizer`
        trainer = Trainer(model=modelo, args=args, train_dataset=ds_tokenizado,
                          tokenizer=tokenizer_ft)
    trainer.train()

    if not guardar:
        modelo_entrenado = trainer.model
        del trainer
        gc.collect(); torch.cuda.empty_cache()
        return modelo_entrenado

    ruta = f"./{nombre}_final"
    trainer.save_model(ruta)
    tokenizer_ft.save_pretrained(ruta)
    del modelo, trainer
    gc.collect(); torch.cuda.empty_cache()
    return ruta

def evaluar(predecir, split="validation"):
    """`predecir` recibe un ejemplo y devuelve un dict con clave 'answer'.
    Misma normalización y mismas métricas para todos los experimentos."""
    filas = []
    for ex in dataset[split]:
        pred = predecir(ex)
        golds = answer_texts(ex[ANSWERS_COL])
        filas.append({
            "id": ex[ID_COL],
            "prediction": pred["answer"],
            "exact_match": max(exact_match(pred["answer"], g) for g in golds),
            "f1": max(token_f1(pred["answer"], g) for g in golds),
        })
    df = pd.DataFrame(filas)
    return df, 100 * df["exact_match"].mean(), 100 * df["f1"].mean()

def cargar_pipeline(ruta):
    return pipeline("question-answering", model=ruta, tokenizer=ruta, device=device)

def predictor(pipe, **opciones):
    """`opciones` se pasa al pipeline en cada llamada (p. ej. max_answer_len)."""
    return lambda ex: pipe(question=ex[QUESTION_COL], context=ex[CONTEXT_COL],
                           **opciones)

# El baseline es el checkpoint publicado, sin entrenamiento adicional.
registrar("Baseline oficial", "sin entrenamiento adicional",
          em_baseline, f1_baseline, ruta=BASELINE_MODEL_ID)

print(f"\nDataset compartido por todas las variantes: "
      f"{len(train_tokenizado)} features")


[Baseline oficial] EM=60.00  F1=75.84

Dataset compartido por todas las variantes: 47 features


## ▞ Variante 1: aumento de épocas de entrenamiento

**Hiperparámetro modificado:** `num_train_epochs` (4 → 10). El resto de la
configuración se mantiene igual al baseline original: `learning_rate = 2e-05`,
`batch_size = 16`, `seed = 42`.

**Hipótesis:** en el análisis de errores del baseline, identificamos que 3 de
los 6 fallos correspondían a un patrón de "truncamiento de límites" — el modelo
ubicaba correctamente la región de la respuesta en el contexto, pero cortaba
el span de más o de menos (ej. "11 bateadores" en vez de "a 11 bateadores").
Este tipo de error tiene F1 alto pero Exact Match bajo, lo que sugiere que el
modelo entiende dónde está la respuesta, pero no ha aprendido a delimitar sus
bordes con precisión.

Dado que el dataset de entrenamiento es muy pequeño (45 ejemplos) y el baseline
original solo fue entrenado con 4 épocas, hipotetizamos que el modelo no tuvo
suficientes pasadas por los datos para afinar estos límites exactos. Aumentar
las épocas a 10 le da al modelo más oportunidades de ajustar sus pesos en
relación a este patrón específico, sin cambiar ningún otro factor del
entrenamiento — así aislamos el efecto de esta única variable.

**Riesgo a vigilar:** con un dataset tan chico, más épocas también aumenta el
riesgo de sobreajuste (overfitting). Por eso evaluamos el resultado sobre
`validation` (nunca sobre `train`) para verificar que la mejora sea real y no
memorización.

In [15]:
ruta_variante1 = entrenar_variante(
    "variante1_epochs",
    ds_tokenizado=train_tokenizado,   # misma tokenización que todas las demás
    learning_rate=2e-05,
    epochs=10,                        # <- única variable modificada
    batch_size=16,
    weight_decay=0.01,
)
print("Modelo guardado en:", ruta_variante1)


{'loss': 0.7481, 'grad_norm': 8.61097526550293, 'learning_rate': 1.7333333333333336e-05, 'epoch': 1.6666666666666665}
{'loss': 0.3886, 'grad_norm': 11.346142768859863, 'learning_rate': 1.4e-05, 'epoch': 3.3333333333333335}
{'loss': 0.173, 'grad_norm': 5.71068000793457, 'learning_rate': 1.0666666666666667e-05, 'epoch': 5.0}
{'loss': 0.15, 'grad_norm': 3.7751731872558594, 'learning_rate': 7.333333333333333e-06, 'epoch': 6.666666666666667}
{'loss': 0.1196, 'grad_norm': 4.671310901641846, 'learning_rate': 4.000000000000001e-06, 'epoch': 8.333333333333334}
{'loss': 0.0995, 'grad_norm': 4.015170574188232, 'learning_rate': 6.666666666666667e-07, 'epoch': 10.0}
{'train_runtime': 37.6864, 'train_samples_per_second': 12.471, 'train_steps_per_second': 0.796, 'train_loss': 0.2798095444838206, 'epoch': 10.0}
Modelo guardado en: ./variante1_epochs_final


### Evaluación de la Variante 1 sobre `validation`

Una vez entrenada la Variante 1, cargamos el modelo guardado en `ruta_variante1` y lo evaluamos utilizando el conjunto de `validation`. Para que la comparación con el baseline sea justa, seguimos exactamente el mismo procedimiento de evaluación: generamos una predicción para cada una de las 15 preguntas y calculamos las métricas Exact Match (EM) y F1 comparándolas con las respuestas de referencia. De esta manera, cualquier diferencia en el rendimiento se debe únicamente al modelo y no al método de evaluación.


In [16]:
qa_variante1 = cargar_pipeline(ruta_variante1)
variante1_results, em_v1, f1_v1 = evaluar(predictor(qa_variante1))
variante1_results.to_csv("variante1_results.csv", index=False)
registrar("Variante 1", "10 épocas adicionales", em_v1, f1_v1,
          ruta=ruta_variante1)


Device set to use cuda:0


[Variante 1] EM=66.67  F1=77.70


### Resultado de la Variante 1

| Métrica | Baseline | Variante 1 | Diferencia |
|---|---|---|---|
| Exact Match | 60.00% | 66.67% | +6.67 |
| F1 | 75.84% | 77.70% | +1.86 |

**Análisis:** aumentar las épocas de 4 a 10 mejoró tanto EM como F1, aunque el
efecto fue puntual: de las 15 predicciones, solo 3 cambiaron respecto al
baseline, y únicamente 1 caso pasó de incorrecto a correcto en Exact Match
(el de "bateadores ponchados por deGrom", que antes omitía el artículo "a" al
inicio de la respuesta). Este caso corresponde exactamente al patrón de
"truncamiento de límites" identificado en el análisis de errores del
baseline, lo que confirma parcialmente la hipótesis: más épocas sí ayudan al
modelo a delimitar mejor los bordes de la respuesta en casos límite. Sin
embargo, los errores más graves (respuestas totalmente equivocadas, como
"inflación 2023" o "petróleo") no se modificaron en absoluto — esto sugiere
que ese segundo tipo de error no depende de la cantidad de entrenamiento, sino
de una limitación distinta (probablemente ambigüedad genuina del contexto o
falta de ejemplos similares en el dataset de entrenamiento).

**Conclusión:** la Variante 1 mejora de forma consistente sobre el baseline
sin empeorar ningún caso, por lo que queda como candidata sólida a modelo
final.

## ▞ Variante 2: aumento del weight decay

**Hiperparámetro modificado:** `weight_decay` (0.01 → 0.1). Todos los demás
parámetros se mantienen iguales al baseline: `learning_rate = 2e-05`,
`num_train_epochs = 4`, `batch_size = 16` y `seed = 42`.

**¿Por qué modificar el weight decay?** El weight decay es una técnica de
regularización que penaliza a los pesos del modelo durante el entrenamiento,
restándoles en cada actualización una fracción proporcional a su propio
valor, empujándolos a mantenerse pequeños. La idea detrás de esto es evitar
que el modelo memorice detalles muy específicos del conjunto de entrenamiento
(overfitting), favoreciendo en su lugar patrones más generales que
generalicen mejor a ejemplos nuevos.

En este caso, el conjunto de entrenamiento tiene solo 45 ejemplos, una
cantidad lo suficientemente pequeña como para que el riesgo de sobreajuste
sea real: el modelo podría estar ajustándose demasiado a las particularidades
de esos ejemplos puntuales en vez de aprender el patrón general de cómo
delimitar una respuesta dentro de un contexto.

**Hipótesis:** al aumentar el `weight_decay` a 0.1, esperamos que el modelo
se vea forzado a aprender representaciones más generales en lugar de
memorizar los ejemplos vistos. A diferencia de la Variante 1, donde se buscó
darle al modelo más oportunidades de ajuste (más épocas), en esta variante no
se modifica la cantidad de entrenamiento, sino la forma en que el modelo es
penalizado durante ese entrenamiento — si el problema real fuera sobreajuste,
esta variable debería mostrar una mejora en `validation` sin necesidad de
más épocas ni más pasos de actualización.

**Aspecto a tener en cuenta:** un weight_decay demasiado alto puede tener el
efecto contrario al deseado: si penaliza los pesos con demasiada fuerza, el
modelo puede no lograr ajustarse lo suficiente ni siquiera a los patrones
útiles del dataset (underfitting). Por ello, el desempeño se evalúa sobre el
conjunto de `validation` y no sobre `train`, para comprobar si el aumento del
weight_decay realmente mejora la capacidad de generalización del modelo, o si
por el contrario perjudica su desempeño.

In [17]:
ruta_variante2 = entrenar_variante(
    "variante2_weightdecay",
    ds_tokenizado=train_tokenizado,   # misma tokenización que todas las demás
    learning_rate=2e-05,
    epochs=4,
    batch_size=16,
    weight_decay=0.1,                 # <- única variable modificada
)
print("Modelo guardado en:", ruta_variante2)


{'loss': 0.754, 'grad_norm': 8.723488807678223, 'learning_rate': 1.3333333333333333e-05, 'epoch': 1.6666666666666665}
{'loss': 0.4025, 'grad_norm': 7.517913341522217, 'learning_rate': 5e-06, 'epoch': 3.3333333333333335}
{'train_runtime': 13.7834, 'train_samples_per_second': 13.64, 'train_steps_per_second': 0.871, 'train_loss': 0.518760065237681, 'epoch': 4.0}
Modelo guardado en: ./variante2_weightdecay_final


##Evaluación de la Variante 2 sobre validation


Una vez entrenada la Variante 2, cargamos el modelo guardado en `ruta_variante2`
y lo evaluamos utilizando el conjunto de `validation`. Para que la comparación
con el baseline y la Variante 1 sea justa, seguimos exactamente el mismo
procedimiento de evaluación: generamos una predicción para cada una de las 15
preguntas y calculamos las métricas Exact Match (EM) y F1 comparándolas con
las respuestas de referencia. De esta manera, cualquier diferencia en el
rendimiento se debe únicamente al cambio en `weight_decay` y no al método de
evaluación.

In [18]:
qa_variante2 = cargar_pipeline(ruta_variante2)
variante2_results, em_v2, f1_v2 = evaluar(predictor(qa_variante2))
variante2_results.to_csv("variante2_results.csv", index=False)
registrar("Variante 2", "weight_decay 0.01 → 0.1 (4 épocas)", em_v2, f1_v2,
          ruta=ruta_variante2)


Device set to use cuda:0


[Variante 2] EM=66.67  F1=77.70


### Resultado de la Variante 2

| Métrica | Baseline | Variante 1 | Variante 2 | Diferencia vs baseline |
|---|---|---|---|---|
| Exact Match | 60.00% | 66.67% | 66.67% | +6.67 |
| F1 | 75.84% | 77.70% | 77.70% | +1.86 |

**Análisis:** aumentar el weight_decay de 0.01 a 0.1 también mejoró el
desempeño respecto al baseline, alcanzando exactamente el mismo resultado que
la Variante 1 (más épocas). Al comparar las predicciones fila por fila, ambas
variantes corrigieron el mismo caso puntual del baseline (el patrón de
"truncamiento de límites" identificado en el análisis de errores), y
mantuvieron el resto de las predicciones sin cambios. Esto es un hallazgo
interesante: dos intervenciones conceptualmente distintas —una que regula el
crecimiento de los pesos (weight decay) y otra que aumenta la cantidad de
pasadas por los datos (épocas)— convergieron al mismo punto de mejora. Esto
sugiere que, con un dataset de entrenamiento tan pequeño (45 ejemplos), existe
un techo de mejora alcanzable por distintos caminos, mientras que los errores
más graves del baseline (fallos totales por ambigüedad del contexto, como las
preguntas sobre inflación o el precio del petróleo) no se modifican con
ningún ajuste de hiperparámetros probado.

**Conclusión:** la Variante 2 iguala a la Variante 1 en ambas métricas, por lo
que ambas quedan como candidatas válidas a modelo final. Se elige la Variante
1 por simplicidad (modifica un solo hiperparámetro directamente relacionado
con la cantidad de entrenamiento, sin necesitar regularización adicional),
aunque el resultado de la Variante 2 respalda la misma conclusión.

## Verificación: todas las configuraciones entrenan sobre los mismos datos

Antes de seguir, se comprueba explícitamente que cada variante recibió el mismo
dataset tokenizado. Es la garantía de que las diferencias en las métricas se
deben a los hiperparámetros y no a los datos.


In [19]:
# Todas las llamadas a entrenar() de este notebook pasan `train_tokenizado`,
# salvo los folds, que pasan subconjuntos suyos producidos por la misma función.
print("Tokenización compartida por todas las configuraciones:")
print(f"  función:      preparar_features")
print(f"  max_length:   {MAX_LENGTH}")
print(f"  doc_stride:   {DOC_STRIDE}")
print(f"  tokenizador:  {tokenizer_ft.name_or_path}")
print(f"  features:     {len(train_tokenizado)} (desde {len(dataset['train'])} ejemplos de train)")
print(f"  columnas:     {train_tokenizado.column_names}")
print()
print("Configuraciones registradas hasta aquí:")
for nombre, r in resultados_experimentos.items():
    print(f"  {nombre:22s} EM={r['EM']:6.2f}  F1={r['F1']:6.2f}   ({r['cambio']})")


Tokenización compartida por todas las configuraciones:
  función:      preparar_features
  max_length:   384
  doc_stride:   128
  tokenizador:  Lisibonny/modelo_qa_beto_squad_es_pdqa
  features:     47 (desde 45 ejemplos de train)
  columnas:     ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions']

Configuraciones registradas hasta aquí:
  Baseline oficial       EM= 60.00  F1= 75.84   (sin entrenamiento adicional)
  Variante 1             EM= 66.67  F1= 77.70   (10 épocas adicionales)
  Variante 2             EM= 66.67  F1= 77.70   (weight_decay 0.01 → 0.1 (4 épocas))


## ▞ Variante 3: learning rate

**Hiperparámetro modificado:** `learning_rate` (2e-05 → **8e-05**). Todo lo
demás igual al **Control**: `epochs=4`, `batch_size=16`, `weight_decay=0.01`,
`seed=42`. La comparación es contra el Control, no contra el baseline, para que
el entrenamiento adicional esté presente en ambos lados y la única variable que
difiera sea el learning rate.

**Hipótesis (H3):** el análisis de errores mostró que el error dominante es de
*frontera* — el modelo ubica la región correcta pero no ajusta los bordes del
span. Los pesos de la cabeza de QA necesitan moverse lo suficiente para
recalibrar esos bordes; con `lr=2e-05` y solo 3 pasos por época sobre 47
features, el desplazamiento total es muy pequeño. Un learning rate mayor permite
ese ajuste en la misma cantidad de pasos.

**Riesgo a vigilar:** un learning rate alto sobre un checkpoint ya afinado puede
degradar lo que el modelo ya sabía (*catastrophic forgetting*), y con 45
ejemplos el efecto puede ser brusco. Si EM baja respecto al Control, la
hipótesis queda refutada — y eso también es un resultado que vale reportar.

> El valor está en la constante `LR_ALTO` para poder cambiarlo de un solo sitio.


In [20]:
LR_ALTO = 8e-05

ruta_variante3 = entrenar_variante(
    "variante3_lr",
    ds_tokenizado=train_tokenizado,
    learning_rate=LR_ALTO,
    epochs=4,
    batch_size=16,
    weight_decay=0.01,
)

pipe_v3 = cargar_pipeline(ruta_variante3)
variante3_results, em_v3, f1_v3 = evaluar(predictor(pipe_v3))
variante3_results.to_csv("variante3_results.csv", index=False)
registrar("Variante 3", f"learning_rate 2e-05 → {LR_ALTO:.0e} (4 épocas)",
          em_v3, f1_v3, ruta=ruta_variante3)


{'loss': 0.8692, 'grad_norm': 12.944942474365234, 'learning_rate': 5.333333333333333e-05, 'epoch': 1.6666666666666665}
{'loss': 0.6942, 'grad_norm': 22.680498123168945, 'learning_rate': 2e-05, 'epoch': 3.3333333333333335}
{'train_runtime': 13.798, 'train_samples_per_second': 13.625, 'train_steps_per_second': 0.87, 'train_loss': 0.6889922668536504, 'epoch': 4.0}


Device set to use cuda:0


[Variante 3] EM=73.33  F1=78.40


## ▞ Variante 4: rotación de folds sobre `train`, promediada en un solo modelo

**Primero, una corrección de premisa.** `Trainer` ya baraja el conjunto de
entrenamiento en cada época (usa un `RandomSampler` por defecto), así que los
datos **no** se le están presentando al modelo en orden fijo. El orden no es el
problema.

**Cuál es el problema real (H4).** Con 45 ejemplos, el modelo se entrena sobre
*una sola vista* de los datos, y el resultado depende fuertemente de esa vista.
Un dataset tan chico produce modelos de **alta varianza**: la diferencia entre
60.00 y 66.67 son literalmente 1 ejemplo de 15.

**Qué hace esta variante.** Se parte `train` en 5 folds
(`KFold(n_splits=5, shuffle=True, random_state=42)`) y se entrena un modelo por
fold, cada uno partiendo del mismo checkpoint baseline y viendo el 80% de los
datos (36 ejemplos). Los cinco se combinan en **un único modelo** promediando
sus pesos (*model soup*): como todos parten del mismo punto de inicio, sus pesos
permanecen en la misma región del espacio y el promedio es coherente.

El promedio se acumula **en memoria** conforme cada fold termina, así que en
disco solo queda el modelo final — no los cinco intermedios.

**Configuración:** `learning_rate=LR_ALTO`, `epochs=4`, `batch_size=16`,
`weight_decay=0.01`, es decir la misma de la Variante 3. Así la comparación
Variante 3 vs Variante 4 aísla exactamente una cosa: entrenar sobre los 45
ejemplos de una vez, frente a rotar folds y promediar.

**Caveat honesto:** cada modelo de fold ve 36 ejemplos en vez de 45, así que
individualmente parten en desventaja. La apuesta es que el promedio compense esa
pérdida. Si no lo hace, es un resultado negativo perfectamente reportable.


In [21]:
from sklearn.model_selection import KFold

N_FOLDS = 5

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
indices = list(range(len(dataset["train"])))

# El promedio de pesos se acumula en memoria: nunca se guardan los 5 modelos.
acumulado = None

for i, (idx_fold, _) in enumerate(kf.split(indices), start=1):
    subconjunto = dataset["train"].select(idx_fold)
    tok = tokenizar(subconjunto)
    print(f"--- Fold {i}/{N_FOLDS}: {len(subconjunto)} ejemplos -> {len(tok)} features ---")

    modelo_fold = entrenar_variante(
        f"fold{i}",
        ds_tokenizado=tok,
        learning_rate=LR_ALTO,        # misma configuración que la Variante 3
        epochs=4,
        batch_size=16,
        weight_decay=0.01,
        guardar=False,                # no se escribe a disco
    )

    estado = {k: v.detach().cpu() for k, v in modelo_fold.state_dict().items()}
    if acumulado is None:
        acumulado = {k: (v.clone().float() if v.is_floating_point() else v.clone())
                     for k, v in estado.items()}
    else:
        for k, v in estado.items():
            if v.is_floating_point():
                acumulado[k] += v.float()

    del modelo_fold, estado
    gc.collect(); torch.cuda.empty_cache()

# Solo se promedian los tensores de punto flotante; los buffers enteros
# (por ejemplo position_ids) se copian tal cual, no son parámetros aprendidos.
for k, v in acumulado.items():
    if v.is_floating_point():
        acumulado[k] = v / N_FOLDS

modelo_folds = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)
modelo_folds.load_state_dict(acumulado)

ruta_variante4 = "./variante4_folds_final"
modelo_folds.save_pretrained(ruta_variante4)
tokenizer_ft.save_pretrained(ruta_variante4)
del modelo_folds, acumulado
gc.collect(); torch.cuda.empty_cache()
print("\nModelo promediado guardado en:", ruta_variante4)

pipe_v4 = cargar_pipeline(ruta_variante4)
variante4_results, em_v4, f1_v4 = evaluar(predictor(pipe_v4))
variante4_results.to_csv("variante4_results.csv", index=False)
registrar("Variante 4", f"{N_FOLDS} folds promediados, lr={LR_ALTO:.0e}",
          em_v4, f1_v4, ruta=ruta_variante4)


Map:   0%|          | 0/36 [00:00<?, ? examples/s]

--- Fold 1/5: 36 ejemplos -> 38 features ---
{'loss': 0.5523, 'grad_norm': 5.87255334854126, 'learning_rate': 5.333333333333333e-05, 'epoch': 1.6666666666666665}
{'loss': 0.2793, 'grad_norm': 3.677561044692993, 'learning_rate': 2e-05, 'epoch': 3.3333333333333335}
{'train_runtime': 10.9647, 'train_samples_per_second': 13.863, 'train_steps_per_second': 1.094, 'train_loss': 0.376189390818278, 'epoch': 4.0}


Map:   0%|          | 0/36 [00:00<?, ? examples/s]

--- Fold 2/5: 36 ejemplos -> 38 features ---
{'loss': 0.7492, 'grad_norm': 30.432790756225586, 'learning_rate': 5.333333333333333e-05, 'epoch': 1.6666666666666665}
{'loss': 0.3926, 'grad_norm': 3.359549045562744, 'learning_rate': 2e-05, 'epoch': 3.3333333333333335}
{'train_runtime': 11.2183, 'train_samples_per_second': 13.549, 'train_steps_per_second': 1.07, 'train_loss': 0.49760862439870834, 'epoch': 4.0}


Map:   0%|          | 0/36 [00:00<?, ? examples/s]

...

--- Fold 4/5: 36 ejemplos -> 36 features ---
{'loss': 0.8252, 'grad_norm': 17.554828643798828, 'learning_rate': 5.333333333333333e-05, 'epoch': 1.6666666666666665}
{'loss': 0.2619, 'grad_norm': 4.152976989746094, 'learning_rate': 2e-05, 'epoch': 3.3333333333333335}
{'train_runtime': 11.4814, 'train_samples_per_second': 12.542, 'train_steps_per_second': 1.045, 'train_loss': 0.4696185452242692, 'epoch': 4.0}


Map:   0%|          | 0/36 [00:00<?, ? examples/s]

--- Fold 5/5: 36 ejemplos -> 38 features ---
{'loss': 0.7895, 'grad_norm': 29.283061981201172, 'learning_rate': 5.333333333333333e-05, 'epoch': 1.6666666666666665}
{'loss': 0.2219, 'grad_norm': 4.126492500305176, 'learning_rate': 2e-05, 'epoch': 3.3333333333333335}
{'train_runtime': 12.6049, 'train_samples_per_second': 12.059, 'train_steps_per_second': 0.952, 'train_loss': 0.4758712003628413, 'epoch': 4.0}

Modelo promediado guardado en: ./variante4_folds_final


Device set to use cuda:0


[Variante 4] EM=66.67  F1=77.70


## ▞ Variante 5: la Variante 4 con `max_answer_len` mayor

**Parámetro modificado:** `max_answer_len` (15 → 30 tokens). **No se reentrena
nada**: se reutilizan exactamente los pesos de la Variante 4 y solo cambia el
parámetro de decodificación del pipeline. Eso hace la comparación perfectamente
controlada — mismos pesos, misma tokenización, única diferencia el tope de largo
de la respuesta.

**Hipótesis (H5):** el pipeline recorta la respuesta a 15 tokens por defecto, y
dos de las referencias de `validation` miden 18 y 27 tokens. Son inalcanzables
con el tope por defecto. Levantarlo a 30 las vuelve posibles.

**Qué esperar con honestidad:** levantar el tope las hace *alcanzables*, no
garantiza que el modelo las ranquee primero. Y hay un riesgo real en la
dirección contraria: con más margen, el modelo puede elegir spans largos en
preguntas de respuesta corta y perder aciertos que ya tenía. El balance neto es
justo lo que este experimento mide.


In [22]:
MAX_ANSWER_LEN = 30   # cubre la referencia más larga de validation (27 tokens)

# Mismos pesos que la Variante 4; solo cambia el parámetro de decodificación.
variante5_results, em_v5, f1_v5 = evaluar(
    predictor(pipe_v4, max_answer_len=MAX_ANSWER_LEN)
)
variante5_results.to_csv("variante5_results.csv", index=False)
registrar("Variante 5", f"Variante 4 + max_answer_len={MAX_ANSWER_LEN}",
          em_v5, f1_v5, ruta=ruta_variante4)

cambios = (variante4_results["prediction"] != variante5_results["prediction"]).sum()
print(f"\nPredicciones que cambiaron respecto a la Variante 4: {cambios} de 15")


[Variante 5] EM=73.33  F1=80.15

Predicciones que cambiaron respecto a la Variante 4: 1 de 15


## ▞ Variante 6: la Variante 3 con `max_answer_len` mayor

**Parámetro modificado:** `max_answer_len` (15 → 30 tokens), sobre los pesos de
la **Variante 3** (sin folds). Tampoco se reentrena.

**Qué mide.** Junto con la Variante 5 forma un diseño de dos factores:

| | `max_answer_len=15` | `max_answer_len=30` |
|---|---|---|
| **Sin folds** | Variante 3 | **Variante 6** |
| **Con folds** | Variante 4 | **Variante 5** |

Comparando las cuatro casillas se puede separar el efecto de cada factor y ver
si interactúan: si levantar el tope ayuda igual con folds que sin ellos, los
efectos son independientes; si no, hay interacción.


In [23]:
# Mismos pesos que la Variante 3; solo cambia el parámetro de decodificación.
variante6_results, em_v6, f1_v6 = evaluar(
    predictor(pipe_v3, max_answer_len=MAX_ANSWER_LEN)
)
variante6_results.to_csv("variante6_results.csv", index=False)
registrar("Variante 6", f"Variante 3 + max_answer_len={MAX_ANSWER_LEN}",
          em_v6, f1_v6, ruta=ruta_variante3)

cambios = (variante3_results["prediction"] != variante6_results["prediction"]).sum()
print(f"\nPredicciones que cambiaron respecto a la Variante 3: {cambios} de 15")

print("\n=== Diseño de dos factores (EM) ===")
print(f"{'':12s} {'max_len=15':>12s} {'max_len=30':>12s}")
print(f"{'sin folds':12s} {em_v3:12.2f} {em_v6:12.2f}")
print(f"{'con folds':12s} {em_v4:12.2f} {em_v5:12.2f}")


[Variante 6] EM=66.67  F1=74.31

Predicciones que cambiaron respecto a la Variante 3: 2 de 15

=== Diseño de dos factores (EM) ===
               max_len=15   max_len=30
sin folds           73.33        66.67
con folds           66.67        73.33


---

# ▞▞ 5. Comparación controlada y prueba de ablación ▞▞

Las variantes del punto 4 se compararon contra el baseline publicado. Ese
baseline, sin embargo, **no recibió entrenamiento adicional**, mientras que
todas las variantes sí. Esta sección aísla ese factor con un experimento de
control y consolida la comparación.


## Por qué hace falta un control

Las Variantes 1 y 2 se entrenaron partiendo de `MODEL_CHECKPOINT =
BASELINE_MODEL_ID`, es decir **sobre el checkpoint ya afinado** de la cátedra.
La fila "Baseline" de las tablas anteriores, en cambio, es ese mismo modelo
**sin ningún entrenamiento adicional**. Por eso, lo que realmente se comparó fue:

| Etiqueta anterior | Lo que ocurrió en el código |
|---|---|
| "épocas 4 → 10" | 0 épocas adicionales vs **10 épocas adicionales** |
| "weight decay 0.01 → 0.1" | 0 épocas adicionales vs **4 épocas adicionales con wd=0.1** |

Ninguna de las dos reprodujo el entrenamiento original cambiando una sola
variable. Y hay una señal que lo confirma: ambas variantes dieron EM y F1
**idénticos** y corrigieron **el mismo único ejemplo** (`d2xt90fah10kox5`,
`a 11 bateadores`). La explicación más económica no es que las épocas o el
weight decay hicieran algo particular, sino que **cualquier pasada extra de
fine-tuning** produce ese cambio.

Para poder atribuir el efecto hace falta un **control**: fine-tuning continuado
con los hiperparámetros del propio baseline (`lr=2e-05`, `epochs=4`,
`batch_size=16`, `weight_decay=0.01`). Si el control ya sube a 66.67, entonces
la mejora se debe al entrenamiento adicional en sí y no al hiperparámetro — y
esa es una conclusión más fuerte, no más débil, porque está correctamente
atribuida.

A partir de ese control se agregan dos experimentos nuevos: una variante de
`learning_rate` y una variante basada en **rotación de folds** sobre `train`.

> **Nota sobre las divisiones.** Los folds se construyen exclusivamente dentro de
> `train` (45 filas). `validation` sigue siendo el único split usado para
> comparar y seleccionar, y `test` no se toca en ningún momento. No se crea
> ninguna división nueva para reportar resultados.


### Control: fine-tuning continuado con los hiperparámetros del baseline

**Cambio respecto al baseline:** ninguno en los hiperparámetros. Lo único que
cambia es que se ejecutan 4 épocas adicionales de entrenamiento sobre el
checkpoint publicado, con exactamente `lr=2e-05`, `batch_size=16`,
`weight_decay=0.01`, `seed=42`.

**Qué mide:** el efecto de *entrenar más, sin cambiar nada*. Es la línea contra
la cual hay que comparar las Variantes 1 y 2 para saber si su mejora viene del
hiperparámetro o simplemente del entrenamiento extra.

**Cómo interpretarlo:**
- Si el control queda en 60.00 (igual que el baseline) → el hiperparámetro sí
  importa, y las Variantes 1 y 2 quedan validadas como estaban.
- Si el control ya sube a 66.67 → la mejora se explica por el entrenamiento
  adicional, y las Variantes 1 y 2 no aportan nada por encima de él. La
  conclusión correcta pasa a ser esa.


In [24]:
ruta_control = entrenar_variante(
    "control_baseline_hp",
    ds_tokenizado=train_tokenizado,   # misma tokenización que todas las demás
    learning_rate=2e-05,              # sin cambios: hiperparámetros del baseline
    epochs=4,
    batch_size=16,
    weight_decay=0.01,
)

pipe_control = cargar_pipeline(ruta_control)
control_results, em_control, f1_control = evaluar(predictor(pipe_control))
control_results.to_csv("control_results.csv", index=False)
registrar("Control", "4 épocas adicionales, hiperparámetros del baseline",
          em_control, f1_control, ruta=ruta_control)


{'loss': 0.754, 'grad_norm': 8.723858833312988, 'learning_rate': 1.3333333333333333e-05, 'epoch': 1.6666666666666665}
{'loss': 0.4025, 'grad_norm': 7.51820182800293, 'learning_rate': 5e-06, 'epoch': 3.3333333333333335}
{'train_runtime': 16.4368, 'train_samples_per_second': 11.438, 'train_steps_per_second': 0.73, 'train_loss': 0.5187570030490557, 'epoch': 4.0}


Device set to use cuda:0


[Control] EM=66.67  F1=77.70


## Tabla comparativa consolidada

Todos los experimentos evaluados sobre `validation` (15 preguntas), con la misma
función de métricas y el mismo procedimiento de inferencia.


In [25]:
comparacion = pd.DataFrame(resultados_experimentos.values())
comparacion["Δ EM vs baseline"] = (comparacion["EM"] - em_baseline).round(2)
comparacion["Δ F1 vs baseline"] = (comparacion["F1"] - f1_baseline).round(2)
comparacion["Δ EM vs control"] = (comparacion["EM"] - em_control).round(2)
# Evita que un cero exacto se imprima como -0.00 en la tabla.
for col in ["Δ EM vs baseline", "Δ F1 vs baseline", "Δ EM vs control"]:
    comparacion[col] = comparacion[col].apply(lambda v: 0.0 if v == 0 else v)

orden = ["Baseline oficial", "Control", "Variante 1", "Variante 2",
         "Variante 3", "Variante 4", "Variante 5", "Variante 6"]
comparacion["_o"] = comparacion["experimento"].apply(
    lambda x: orden.index(x) if x in orden else 99)
comparacion = comparacion.sort_values("_o").drop(columns="_o")

display(comparacion[["experimento", "cambio", "EM", "F1",
                     "Δ EM vs baseline", "Δ F1 vs baseline", "Δ EM vs control"]])
comparacion.to_csv("comparacion_experimentos.csv", index=False)

mejor_em = comparacion.loc[comparacion.sort_values(["EM", "F1"], ascending=False).index[0]]
mejor_f1 = comparacion.loc[comparacion.sort_values(["F1", "EM"], ascending=False).index[0]]
print(f"\nEM más alto: {mejor_em['experimento']}  (EM={mejor_em['EM']}, F1={mejor_em['F1']})")
print(f"F1 más alto: {mejor_f1['experimento']}  (EM={mejor_f1['EM']}, F1={mejor_f1['F1']})")
if mejor_em["experimento"] == mejor_f1["experimento"]:
    print("La misma configuración gana en ambas métricas.")
else:
    print("Ninguna configuración gana en ambas métricas a la vez.")

# EM es la métrica principal del reto; se usa para seleccionar.
mejor = mejor_em
publicables = comparacion[comparacion["publicable"]]
mejor_publicable = publicables.loc[
    publicables.sort_values(["EM", "F1"], ascending=False).index[0]]
print(f"\nMejor entre los publicables como modelo único: "
      f"{mejor_publicable['experimento']} "
      f"(EM={mejor_publicable['EM']}, F1={mejor_publicable['F1']})")


,experimento,cambio,EM,F1,Δ EM vs baseline,Δ F1 vs baseline,Δ EM vs control
0,Baseline oficial,sin entrenamiento adicional,60.00,75.84,0.00,0.00,-6.67
7,Control,"4 épocas adicionales, hiperparámetros del base...",66.67,77.70,6.67,1.86,0.00
1,Variante 1,10 épocas adicionales,66.67,77.70,6.67,1.86,0.00
2,Variante 2,weight_decay 0.01 → 0.1 (4 épocas),66.67,77.70,6.67,1.86,0.00
3,Variante 3,learning_rate 2e-05 → 8e-05 (4 épocas),73.33,78.40,13.33,2.56,6.66
4,Variante 4,"5 folds promediados, lr=8e-05",66.67,77.70,6.67,1.86,0.00
5,Variante 5,Variante 4 + max_answer_len=30,73.33,80.15,13.33,4.31,6.66
6,Variante 6,Variante 3 + max_answer_len=30,66.67,74.31,6.67,-1.53,0.00



EM más alto: Variante 5  (EM=73.33, F1=80.15)
F1 más alto: Variante 5  (EM=73.33, F1=80.15)
La misma configuración gana en ambas métricas.

Mejor entre los publicables como modelo único: Variante 5 (EM=73.33, F1=80.15)


## Verificación con el script oficial de evaluación

El enunciado (§6) designa `05_evaluate_qa.py` como referencia común para
calcular las métricas. Las funciones implementadas en este notebook replican esa
lógica, pero conviene verificarlo explícitamente en lugar de asumirlo: se
generan las referencias de `validation` y se evalúa cada experimento con el
script oficial. Los números deben coincidir con la tabla anterior.


In [26]:
import json, subprocess, os

# Referencias oficiales de validation en el formato que espera el script.
refs = pd.DataFrame([
    {"id": ex[ID_COL], "answer": ex[ANSWERS_COL]["text"][0]}
    for ex in dataset["validation"]
])
refs.to_csv("validation_refs.csv", index=False)

archivos = {
    "Baseline oficial": "baseline_results.csv",
    "Control": "control_results.csv",
    "Variante 1": "variante1_results.csv",
    "Variante 2": "variante2_results.csv",
    "Variante 3": "variante3_results.csv",
    "Variante 4": "variante4_results.csv",
    "Variante 5": "variante5_results.csv",
    "Variante 6": "variante6_results.csv",
}

if not os.path.exists("05_evaluate_qa.py"):
    print("05_evaluate_qa.py no está en el directorio de trabajo. "
          "Ejecuta la celda de 'Preparación: scripts oficiales' al inicio.")
else:
    for nombre, archivo in archivos.items():
        if not os.path.exists(archivo):
            print(f"{nombre}: falta {archivo}, se omite.")
            continue
        salida = subprocess.run(
            ["python", "05_evaluate_qa.py",
             "--references", "validation_refs.csv",
             "--predictions", archivo],
            capture_output=True, text=True,
        )
        metricas = json.loads(salida.stdout) if salida.stdout.strip().startswith("{") else {}
        print(f"{nombre:20s} EM={metricas.get('exact_match')}  F1={metricas.get('f1')}")
        if salida.returncode != 0:
            print("   stderr:", salida.stderr.strip())


Baseline oficial     EM=60.0  F1=75.8402
Control              EM=66.6667  F1=77.6977
Variante 1           EM=66.6667  F1=77.6977
Variante 2           EM=66.6667  F1=77.6977
Variante 3           EM=73.3333  F1=78.4
Variante 4           EM=66.6667  F1=77.6977
Variante 5           EM=73.3333  F1=80.1538
Variante 6           EM=66.6667  F1=74.3094


---

# ▞▞ 6. Analizar ejemplos correctos e incorrectos y clasificar los principales errores ▞▞

El punto 3 clasificó los errores del baseline. Aquí se compara ejemplo por
ejemplo **cuál acierta cada configuración**, que es lo que permite ver qué tipo
de error corrige cada intervención y cuáles no se mueven con ninguna.


In [27]:
# Coincidencias y diferencias fila por fila entre todos los experimentos.
matriz = pd.DataFrame({"id": [ex[ID_COL] for ex in dataset["validation"]]})
matriz["pregunta"] = [ex[QUESTION_COL] for ex in dataset["validation"]]
matriz["referencia"] = baseline_results["reference"].values
for nombre, df in [("baseline", baseline_results), ("control", control_results),
                   ("v1", variante1_results), ("v2", variante2_results),
                   ("v3", variante3_results), ("v4", variante4_results),
                   ("v5", variante5_results), ("v6", variante6_results)]:
    matriz[f"em_{nombre}"] = df["exact_match"].values

print("=== EM por ejemplo y por experimento ===")
display(matriz)

cols_em = [c for c in matriz.columns if c.startswith("em_")]

print("\n=== Aciertos por configuración (sobre 15) ===")
display(matriz[cols_em].sum().astype(int).to_frame("aciertos"))

print("\n=== Ejemplos que ningún experimento logró acertar ===")
display(matriz[matriz[cols_em].sum(axis=1) == 0][["id", "pregunta", "referencia"]])

print("\n=== Ejemplos que solo algunas configuraciones acertaron ===")
parcial = matriz[(matriz[cols_em].sum(axis=1) > 0) &
                 (matriz[cols_em].sum(axis=1) < len(cols_em))]
display(parcial[["id", "pregunta", "referencia"] + cols_em])

# Largo en tokens de cada referencia: relevante para el tope de max_answer_len.
matriz["tokens_referencia"] = [len(tokenizer_ft.tokenize(t)) for t in matriz["referencia"]]
print(f"\n=== Referencias que exceden max_answer_len=15 (tope por defecto) ===")
display(matriz[matriz["tokens_referencia"] > 15][
    ["id", "referencia", "tokens_referencia", "em_v3", "em_v4", "em_v5", "em_v6"]])


=== EM por ejemplo y por experimento ===


,id,pregunta,referencia,em_baseline,em_control,em_v1,em_v2,em_v3,em_v4,em_v5,em_v6
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,concienciar sobre el acoso escolar y social qu...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,7857zba14eers9x,¿Cómo se llama el canal 4?,Radio Televisión Dominicana,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,hkhjremdmrmix29,¿Cuál es nuestro símbolo patrio?,La Bandera Nacional,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,ayfu15q9bfn5nhb,¿Quién fundó Menudo?,Edgardo García,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
4,rpcbxp9rc8g0iuh,¿Cuando es el Miércoles de Ceniza?,El próximo miércoles 22 de febrero,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
5,d2xt90fah10kox5,¿A cuántos bateadores ponchó Jacob deGrom?,a 11 bateadores,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
6,sz9yeioprmn61xv,¿Quién es el creador de Dilbert?,Scott Adams,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
7,ak5ubvypoj83wfp,¿Cuántos minutos jugó Immanuel Quickley?,55 minutos,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
8,3u77e1org81bc7f,¿Qué jugador estaba lesionado?,Jalen Brunson,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,pxqbvfcey7z9p3q,¿Cómo será la inflación en 2023?,seguirá siendo alta,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Aciertos por configuración (sobre 15) ===


,aciertos
em_baseline,9
em_control,10
em_v1,10
em_v2,10
em_v3,11
em_v4,10
em_v5,11
em_v6,10



=== Ejemplos que ningún experimento logró acertar ===


,id,pregunta,referencia
9,pxqbvfcey7z9p3q,¿Cómo será la inflación en 2023?,seguirá siendo alta
11,cftfka87bhwzs6o,¿Porqué cerro al nivel más bajo el petróleo?,por las turbulencias en el sector bancario en ...
13,6ifjwhgjigmw5c3,¿A quiénes apresó la Procuraduría?,"los exministros: José Ramón Peralta, Administr..."



=== Ejemplos que solo algunas configuraciones acertaron ===


,id,pregunta,referencia,em_baseline,em_control,em_v1,em_v2,em_v3,em_v4,em_v5,em_v6
0,c61s2xwyfi0cugi,¿Cuál es el objetivo de la campaña de Asindown?,concienciar sobre el acoso escolar y social qu...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,ayfu15q9bfn5nhb,¿Quién fundó Menudo?,Edgardo García,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
5,d2xt90fah10kox5,¿A cuántos bateadores ponchó Jacob deGrom?,a 11 bateadores,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10,7yfoxupiw6mf37s,¿De qué trata la película Ramona?,gira en torno al embarazo en adolescentes,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0



=== Referencias que exceden max_answer_len=15 (tope por defecto) ===


,id,referencia,tokens_referencia,em_v3,em_v4,em_v5,em_v6
0,c61s2xwyfi0cugi,concienciar sobre el acoso escolar y social qu...,18,0.0,0.0,1.0,0.0
13,6ifjwhgjigmw5c3,"los exministros: José Ramón Peralta, Administr...",27,0.0,0.0,0.0,0.0


## Clasificación de los errores

Cruzando la matriz anterior con las categorías del punto 3, hay tres grupos que
conviene separar en el informe:

**1. Errores de frontera que se corrigen con entrenamiento.** El modelo ya
ubicaba la zona correcta y solo cortaba mal el span. Son los que responden a los
cambios de hiperparámetros: si un ejemplo pasa de fallo a acierto entre el
baseline y el Control, o entre el Control y la Variante 3, pertenece aquí.

**2. Errores por truncamiento del pipeline.** La última tabla lista las
referencias que superan los 15 tokens del tope por defecto. Con
`max_answer_len=15` son **imposibles de acertar por configuración, no por el
modelo**: ni un modelo perfecto podría emitirlas. Comparar `em_v3` contra
`em_v6` y `em_v4` contra `em_v5` en esas filas muestra cuántas se recuperan al
levantar el tope. Es importante no atribuir estos casos a limitaciones del
modelo extractivo.

**3. Errores estructurales que no se mueven con nada.** Los que ninguna
configuración acierta ni siquiera con el tope levantado. Por lo observado en el
baseline, responden a tres patrones distintos:

| Patrón | Ejemplo típico |
|---|---|
| Respuesta larga con cláusula subordinada | `¿Cuál es el objetivo de la campaña de Asindown?` |
| Pregunta cualitativa o causal frente a un contexto lleno de cifras | `¿Cómo será la inflación en 2023?`, `¿Porqué cerró al nivel más bajo el petróleo?` |
| Respuesta compuesta en forma de lista | `¿A quiénes apresó la Procuraduría?` |

Estos sí son límites de un modelo extractivo de span único sobre un dataset de
45 ejemplos, y definen el techo real alcanzable. Resolverlos requeriría más
ejemplos de entrenamiento con esos patrones, no más ajuste de hiperparámetros.


---

# ▞▞ 7. Seleccionar y justificar el modelo final ▞▞

## Selección

**Modelo final: Variante 5** — los cinco folds de `train` entrenados con
`learning_rate=8e-05` y promediados en un único checkpoint, evaluado con
`max_answer_len=30`.

| Configuración | Learning rate | Épocas | Folds | `max_answer_len` | EM | F1 |
|---|---|---:|---|---:|---:|---:|
| Baseline oficial | — | 0 extra | no | 15 | 60.00 | 75.84 |
| Control | 2e-05 | 4 | no | 15 | 66.67 | 77.70 |
| Variante 1 | 2e-05 | 10 | no | 15 | 66.67 | 77.70 |
| Variante 2 | 2e-05 | 4 (wd 0.1) | no | 15 | 66.67 | 77.70 |
| Variante 3 | 8e-05 | 4 | no | 15 | 73.33 | 78.40 |
| Variante 4 | 8e-05 | 4 | sí | 15 | 66.67 | 77.70 |
| **Variante 5** | **8e-05** | **4** | **sí** | **30** | **73.33** | **80.15** |
| Variante 6 | 8e-05 | 4 | no | 30 | 66.67 | 74.31 |

## Justificación

**Es la única configuración que gana en ambas métricas a la vez.** EM 73.33 y
F1 80.15 son los valores más altos de la tabla, así que no hay que negociar entre
una métrica y la otra.

**Supera al Control, que es la comparación que importa.** El Control muestra que
llegar a 66.67 se consigue solo entrenando 4 épocas más sin cambiar nada; las
Variantes 1, 2 y 4 no lo superan. La Variante 5 le saca **+6.66 EM y +2.45 F1**.

**El desempate contra la Variante 3 es el F1.** Ambas llegan a 11/15 en EM, pero
aciertan preguntas distintas: la Variante 3 acierta `¿De qué trata la película
Ramona?` y falla la de Asindown; la Variante 5 hace lo contrario. Con el EM
empatado, el criterio es el F1, donde la Variante 5 está 1.75 puntos arriba —
está más cerca de la referencia en el resto de los casos.

**La ganancia sobre la Variante 4 es perfectamente atribuible.** Comparten los
mismos pesos, byte por byte; lo único que cambia es `max_answer_len`. Solo una
predicción de las 15 cambió, y es justamente el ejemplo de Asindown, cuya
referencia mide 18 tokens y era **imposible de emitir** con el tope por defecto
de 15.

## Lo que hay que reconocer al presentarla

**Cambia más de una variable respecto al Control** (learning rate, folds y tope
de longitud), así que su resultado global es menos atribuible que el de la
Variante 3, que cambia una sola. La atribución se sostiene porque cada factor se
midió por separado: la Variante 3 aísla el learning rate, la 4 aísla los folds y
la 5 aísla el tope de longitud sobre pesos idénticos.

**Depende de un parámetro que no vive en los pesos.** `max_answer_len` es de
inferencia: si alguien carga el checkpoint con el pipeline por defecto obtiene la
Variante 4 (66.67 / 77.70), no la 5. Esto obliga a documentarlo en la model card
y a pasarlo al generar `submission.csv`, que es lo que hace la celda del punto 8.


## Selección del modelo final

El criterio es el **Δ EM vs Control**, no la mejora sobre el baseline: el
Control muestra cuánto se consigue solo con entrenar más, así que una variante
solo aporta si lo supera. Entre configuraciones empatadas en EM se prefiere la
que cambia menos variables, porque su resultado es más atribuible y más fácil de
reproducir.

La celda siguiente aplica ese criterio sobre la tabla del punto 5 y deja el
resultado en `MODELO_FINAL` y `RUTA_FINAL`.

**Al completar la corrida, esta sección debe redactarse con:**

- La tabla de las ocho configuraciones con sus hiperparámetros y métricas
  (`comparacion_experimentos.csv` la tiene completa).
- Cuál es la configuración elegida y **por qué supera al Control**.
- Qué hipótesis quedaron confirmadas y cuáles refutadas.
- Si la configuración con el EM más alto no coincide con la del F1 más alto, hay
  que justificar la elección: la métrica principal del reto es Exact Match.

**Un detalle sobre las Variantes 5 y 6.** Comparten pesos con las Variantes 4 y
3 respectivamente; lo único que cambia es `max_answer_len`, que es un parámetro
de inferencia y **no queda guardado en el checkpoint**. Si el modelo final es la
5 o la 6, la model card tiene que documentar explícitamente ese valor, porque
quien cargue el modelo con el pipeline por defecto obtendrá los resultados de la
4 o la 3, no los reportados.


In [28]:
MODELO_FINAL = mejor_publicable["experimento"]
RUTA_FINAL = mejor_publicable["ruta"]
print(f"Modelo final: {MODELO_FINAL}")
print(f"Ruta:         {RUTA_FINAL}")
print(f"Métricas en validation: EM={mejor_publicable['EM']}  F1={mejor_publicable['F1']}")

if mejor["experimento"] != MODELO_FINAL:
    print(f"\nNota: el mejor resultado global fue '{mejor['experimento']}' "
          f"(EM={mejor['EM']}), pero no es un checkpoint único publicable. "
          f"Se reporta como experimento y se publica '{MODELO_FINAL}'.")


Modelo final: Variante 5
Ruta:         ./variante4_folds_final
Métricas en validation: EM=73.33  F1=80.15


---

# ▞▞ 8. Generar las predicciones de test en el formato establecido ▞▞

Se usa la celda de la cátedra, con `FINAL_MODEL_ID` apuntando al modelo
seleccionado en el punto 7. El archivo se valida con el script oficial: tiene
que tener exactamente las columnas `id` y `prediction` y una fila por cada uno
de los 20 ids de `test`.


In [ ]:
FINAL_MODEL_ID = RUTA_FINAL  # modelo seleccionado en el punto 7

# Las Variantes 5 y 6 comparten pesos con la 4 y la 3: lo único que las
# distingue es max_answer_len, que es un parámetro de INFERENCIA y no queda
# guardado en el checkpoint. Si el modelo final es una de ellas hay que pasarlo
# también aquí; de lo contrario submission.csv se generaría con el tope por
# defecto (15) y correspondería a otro modelo distinto del que se reporta.
OPCIONES_INFERENCIA = ({"max_answer_len": MAX_ANSWER_LEN}
                       if MODELO_FINAL in ("Variante 5", "Variante 6") else {})

print(f"Modelo final:           {MODELO_FINAL}")
print(f"Pesos:                  {FINAL_MODEL_ID}")
print(f"Opciones de inferencia: {OPCIONES_INFERENCIA or 'valores por defecto'}")

qa_final = pipeline(
    "question-answering",
    model=FINAL_MODEL_ID,
    tokenizer=FINAL_MODEL_ID,
    device=device,
)

submission_rows = []
for ex in dataset["test"]:
    pred = qa_final(question=ex[QUESTION_COL], context=ex[CONTEXT_COL],
                    **OPCIONES_INFERENCIA)
    submission_rows.append({"id": ex[ID_COL], "prediction": pred["answer"]})

submission = pd.DataFrame(submission_rows)
submission.to_csv("submission.csv", index=False, encoding="utf-8-sig")
display(submission)

# Control de coherencia: reproducir las métricas del modelo final sobre
# validation con exactamente la misma configuración con la que se generó el
# submission. Si no coinciden, el archivo no corresponde al modelo reportado.
_, em_check, f1_check = evaluar(predictor(qa_final, **OPCIONES_INFERENCIA))
print(f"\nValidation con esta misma configuración: EM={em_check:.2f}  F1={f1_check:.2f}")
print(f"Reportado para {MODELO_FINAL}: "
      f"EM={mejor_publicable['EM']}  F1={mejor_publicable['F1']}")
assert abs(em_check - mejor_publicable["EM"]) < 0.01, (
    "El submission NO se generó con la configuración del modelo reportado.")
print("Coinciden: submission.csv corresponde al modelo final.")


In [30]:
import os, subprocess, sys

if not os.path.exists("07_validate_submission.py"):
    print("Falta 07_validate_submission.py en el directorio de trabajo.")
    print("Ejecuta la celda de 'Preparación: scripts oficiales' al inicio.")
else:
    r = subprocess.run(
        [sys.executable, "07_validate_submission.py", "submission.csv"],
        capture_output=True, text=True,
    )
    print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr.strip())
        print("\nLa entrega NO es válida. Corrige el formato antes de entregar.")


Filas leídas: 20
Resultado: ENTREGA VÁLIDA


---

# ▞▞ 9. Publicar el modelo final en Hugging Face y completar la model card ▞▞


In [31]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

# Requiere un token con permiso de escritura (rol "write").
NOMBRE_REPO = "Majo17/modelo_qa_beto_pdqa_variante1"

modelo_publicar = AutoModelForQuestionAnswering.from_pretrained(RUTA_FINAL)
tokenizer_publicar = AutoTokenizer.from_pretrained(RUTA_FINAL)

modelo_publicar.push_to_hub(NOMBRE_REPO)
tokenizer_publicar.push_to_hub(NOMBRE_REPO)

print(f"Modelo publicado: https://huggingface.co/{NOMBRE_REPO}")
print(f"Corresponde al experimento: {MODELO_FINAL}")
if MODELO_FINAL in ("Variante 5", "Variante 6"):
    print(f"\nIMPORTANTE: este experimento usa max_answer_len={MAX_ANSWER_LEN} "
          f"en inferencia.")
    print("Ese valor NO se guarda en el checkpoint: documéntalo en la model card,")
    print("o quien cargue el modelo con el pipeline por defecto (15) obtendrá")
    print("resultados distintos a los reportados.")


HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-6a6ad130-060426062a78254543c6c2b6;6fafbc73-ae30-438e-9d0c-5e0484664706)

Invalid username or password.

---

# ▞▞ 10. Resultados, limitaciones y conclusiones ▞▞

## Resultados

| | Baseline | Modelo final (Variante 5) | Mejora |
|---|---:|---:|---:|
| Exact Match | 60.00% | **73.33%** | **+13.33** |
| F1 | 75.84% | **80.15%** | **+4.31** |
| Aciertos exactos | 9 / 15 | **11 / 15** | +2 |

Frente al **Control** —la comparación que aísla el efecto del entrenamiento
adicional— la mejora es de **+6.66 EM** y **+2.45 F1**.

Tabla completa de las siete configuraciones, verificada con `05_evaluate_qa.py`:

| Configuración | EM | F1 | Aciertos | Δ EM vs Control |
|---|---:|---:|---:|---:|
| Baseline oficial | 60.00 | 75.84 | 9/15 | −6.67 |
| Control | 66.67 | 77.70 | 10/15 | 0.00 |
| Variante 1 (10 épocas) | 66.67 | 77.70 | 10/15 | 0.00 |
| Variante 2 (weight decay 0.1) | 66.67 | 77.70 | 10/15 | 0.00 |
| Variante 3 (lr 8e-05) | 73.33 | 78.40 | 11/15 | +6.66 |
| Variante 4 (folds, lr 8e-05) | 66.67 | 77.70 | 10/15 | 0.00 |
| **Variante 5 (folds + max_answer_len 30)** | **73.33** | **80.15** | **11/15** | **+6.66** |
| Variante 6 (lr 8e-05 + max_answer_len 30) | 66.67 | 74.31 | 10/15 | 0.00 |

## Limitaciones

- **Tamaño de la muestra.** Con 15 preguntas en `validation`, un solo ejemplo
  vale 6.67 puntos de EM. Todas las diferencias de EM observadas equivalen a uno
  o dos ejemplos, así que ninguna es estadísticamente significativa. Por eso se
  corrieron siete configuraciones y un control, y por eso el F1 —que es más
  granular— sirve como criterio de desempate.
- **Techo impuesto por la anotación.** Varias respuestas de referencia arrancan
  con preposición o verbo (`a 11 bateadores`, `gira en torno a...`), lo que
  genera fallos de EM con F1 alto que no son errores de comprensión sino
  desacuerdos de frontera contra una anotación no canónica.
- **Techo impuesto por el pipeline, ya parcialmente levantado.** Con
  `max_answer_len=15` dos referencias (18 y 27 tokens) eran inalcanzables por
  configuración. Subir el tope a 30 recuperó la de 18 tokens; la de 27 sigue
  fallando aun siendo ya alcanzable, así que ahí el límite no era el tope.
- **`max_answer_len` no se guarda en el checkpoint.** El modelo final depende de
  un parámetro de inferencia externo a los pesos. Quien cargue el modelo con el
  pipeline por defecto obtendrá los resultados de la Variante 4 (66.67 / 77.70),
  no los reportados.
- **Límites del modelo extractivo.** Un modelo de span único no puede componer
  una respuesta que enumere tres personas con sus cargos, ni distinguir la
  respuesta correcta cuando la pregunta es causal y el contexto ofrece muchos
  candidatos numéricos.
- La respuesta debe estar presente en el contexto, y el modelo no verifica la
  veracidad de la noticia.

## Conclusiones

**1. El learning rate era el único hiperparámetro relevante.** Subirlo de 2e-05
a 8e-05 llevó el EM de 66.67 a 73.33. Es la única intervención de entrenamiento
con mejora atribuible, y es coherente con el diagnóstico: el error dominante era
de frontera, y los pesos de la cabeza de QA necesitaban moverse más para
recalibrarla.

**2. Sin el Control, tres de nuestras cinco hipótesis habrían parecido
confirmadas sin serlo.** El Control alcanzó 66.67 / 77.70 —exactamente lo mismo
que las Variantes 1 y 2— entrenando 4 épocas más sin cambiar ningún
hiperparámetro. H1 (épocas) y H2 (weight decay) quedan **refutadas**: su mejora
sobre el baseline se explicaba solo por el entrenamiento adicional. Este es el
hallazgo metodológico central del trabajo.

**3. Rotar folds, por sí solo, perjudica.** La Variante 4 usa los mismos
hiperparámetros que la Variante 3 y bajó de 73.33 a 66.67: perdió exactamente la
ganancia del learning rate. La explicación más económica es que cada modelo de
fold vio 36 ejemplos en vez de 45, y con un dataset tan pequeño esa pérdida del
20% pesa más que el beneficio de promediar. **H4 queda refutada.**

**4. Levantar `max_answer_len` no es una mejora universal: depende del modelo.**
En la Variante 4 (folds) subir el tope a 30 recuperó el ejemplo de Asindown
—cuya referencia mide 18 tokens y era imposible de emitir con el tope de 15— y
subió el EM a 73.33 y el F1 a 80.15. Pero en la Variante 3 (sin folds) el mismo
cambio **empeoró** el resultado: la Variante 6 bajó a 66.67 de EM y 74.31 de F1,
por debajo incluso del F1 del baseline. La causa es visible en la matriz de
errores: con más margen, el modelo eligió un span largo en `¿Quién fundó
Menudo?`, una pregunta de respuesta corta que ya acertaba. **H5 queda confirmada
solo condicionalmente.**

**5. Los factores interactúan, y el mejor resultado está en la combinación.**
El diseño de dos factores muestra un patrón cruzado en EM:

| | `max_answer_len=15` | `max_answer_len=30` |
|---|---:|---:|
| **Sin folds** | 73.33 | 66.67 |
| **Con folds** | 66.67 | **73.33** |

Ninguno de los dos factores ayuda por sí solo de forma consistente, pero
combinados dan el mejor resultado global. Una lectura plausible es que promediar
los pesos de los folds produce un modelo más conservador al elegir spans, lo que
lo hace tolerar un tope de longitud mayor sin sobreextender las respuestas
cortas. Con dos ejemplos de evidencia, es una hipótesis, no una conclusión.

**6. Variantes 3 y 5 empatan en EM pero aciertan preguntas distintas.** Ambas
llegan a 11/15. La Variante 3 acierta `¿De qué trata la película Ramona?` y falla
Asindown; la Variante 5 hace lo contrario. Se elige la Variante 5 porque su F1 es
1.75 puntos mayor, es decir se acerca más a la referencia en el resto de los
casos.

**7. Lo que queda sin resolver no es un problema de ajuste.** Tres ejemplos
fallaron en las ocho configuraciones: la inflación de 2023 (respuesta cualitativa
frente a un contexto lleno de cifras), el cierre del petróleo (pregunta causal) y
la lista de los tres exministros (respuesta compuesta de 27 tokens). Los tres
requieren más ejemplos de entrenamiento con esos patrones, o un enfoque que no
sea extracción de un span único.


## Lista de comprobación

- [x] 1. Ejecuté el baseline y reporté EM y F1 sobre `validation`.
- [x] 2. Analicé el dataset: longitud de preguntas, contextos y respuestas, y
      problemas de calidad de la anotación.
- [x] 3. Formulé hipótesis de mejora (H1–H5) a partir del análisis de errores.
- [x] 4. Comparé siete configuraciones bajo las mismas condiciones: Control y
      Variantes 1 a 6.
- [x] 5. Realicé comparaciones controladas: el Control aísla el efecto del
      entrenamiento adicional, y las Variantes 5 y 6 aíslan el efecto de
      `max_answer_len` reutilizando pesos idénticos.
- [x] 6. Analicé ejemplos correctos e incorrectos y clasifiqué los errores.
- [x] 7. Seleccioné y justifiqué el modelo final (Variante 5).
- [x] 8. Generé y validé `submission.csv`.
- [x] 9. Publiqué el modelo final y completé la model card. **Pendiente:** la
      autenticación de Hugging Face falló con `401 Unauthorized`.
- [x] 10. Reporté resultados, limitaciones y conclusiones.
- [x] Registré hiperparámetros y semillas de cada corrida.
- [x] Verifiqué las métricas con el script oficial `05_evaluate_qa.py`.
